# API and GPU prerequisite stage
This stage must pass before any model is downloaded.

In [1]:
import subprocess, json
from pathlib import Path
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,index', '--format=csv,noheader'], capture_output=True, text=True)
print(gpu.stdout.strip())
assert gpu.returncode == 0, gpu.stderr
devices = [line for line in gpu.stdout.splitlines() if line.strip()]
assert len(devices) >= 2 and all('T4' in line for line in devices[:2]), devices
print('GPU_STAGE_PASS', len(devices), 'Tesla T4 devices')
state = Path('/kaggle/working/.qwen-state')
state.mkdir(parents=True, exist_ok=True)
(state / 'gpu_stage.json').write_text(json.dumps({'stage': 'gpu_stage', 'devices': devices}, indent=2) + '\n')


Tesla T4, 0
Tesla T4, 1
GPU_STAGE_PASS 2 Tesla T4 devices


84

In [2]:
import json, os, re, secrets, subprocess, sys, time, urllib.request
from pathlib import Path

work = Path('/kaggle/working')
key = os.environ.get('QWEN_API_KEY') or 'test-api-key-change-after-test'
config_path = work / 'qwen_api.json'

# Ensure persistence module is installed in working dir
(work / 'persistence.py').write_text('''"""Small, restart-safe checkpoints for Kaggle working directory."""
from __future__ import annotations
import json, os
from pathlib import Path

ROOT = Path(os.environ.get("QWEN_WORKDIR", "/kaggle/working"))
STATE = ROOT / ".qwen-state"

def mark(stage: str, **details: object) -> None:
    STATE.mkdir(parents=True, exist_ok=True)
    target = STATE / f"{stage}.json"
    temporary = target.with_suffix(".tmp")
    temporary.write_text(json.dumps({"stage": stage, **details}, indent=2) + "\\n")
    temporary.replace(target)

def completed(stage: str) -> bool:
    return (STATE / f"{stage}.json").is_file()

def show() -> list[str]:
    return sorted(path.stem for path in STATE.glob("*.json")) if STATE.exists() else []
''')

# Clean up any stale servers
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(1)

server_code = r'''
import json, os, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
MODEL = os.environ.get('MODEL_NAME', 'qwen-staged-mock')
KEY = os.environ['QWEN_API_KEY']
class Handler(BaseHTTPRequestHandler):
    def log_message(self, *args): pass
    def send_json(self, status, value):
        data = json.dumps(value).encode()
        self.send_response(status); self.send_header('Content-Type', 'application/json'); self.send_header('Content-Length', str(len(data))); self.end_headers(); self.wfile.write(data)
    def authorized(self):
        return self.headers.get('Authorization') == 'Bearer ' + KEY
    def do_GET(self):
        if self.path == '/health': return self.send_json(200, {'status': 'ok', 'model': MODEL})
        if self.path == '/v1/models' and self.authorized(): return self.send_json(200, {'object': 'list', 'data': [{'id': MODEL, 'object': 'model', 'owned_by': 'kaggle'}]})
        self.send_json(401 if self.path.startswith('/v1/') else 404, {'detail': 'invalid API key' if self.path.startswith('/v1/') else 'not found'})
    def do_POST(self):
        if self.path != '/v1/chat/completions' or not self.authorized(): return self.send_json(401, {'detail': 'invalid API key'})
        size = int(self.headers.get('Content-Length', '0')); request = json.loads(self.rfile.read(size) or '{}')
        text = 'Staged Qwen API is healthy.'
        if request.get('stream'):
            body = ''.join('data: ' + json.dumps({'id': 'chatcmpl-staged', 'object': 'chat.completion.chunk', 'choices': [{'index': 0, 'delta': {'content': token + ' '}, 'finish_reason': None}]}) + '\n\n' for token in text.split()) + 'data: [DONE]\n\n'
            data = body.encode(); self.send_response(200); self.send_header('Content-Type', 'text/event-stream'); self.send_header('Content-Length', str(len(data))); self.end_headers(); self.wfile.write(data); return
        self.send_json(200, {'id': 'chatcmpl-staged', 'object': 'chat.completion', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': text}, 'finish_reason': 'stop'}], 'model': request.get('model', MODEL)})
ThreadingHTTPServer(('0.0.0.0', 8000), Handler).serve_forever()
'''
env = {**os.environ, 'QWEN_API_KEY': key, 'MODEL_NAME': 'qwen-staged-mock'}
server = subprocess.Popen([sys.executable, '-c', server_code], env=env, cwd=work)
pid = server.pid
time.sleep(2)

def call(path, method='GET', payload=None, auth=False):
    data = None if payload is None else json.dumps(payload).encode()
    headers = {'Authorization': f'Bearer {key}'} if auth else {}
    if data: headers['Content-Type'] = 'application/json'
    request = urllib.request.Request('http://127.0.0.1:8000' + path, data=data, headers=headers, method=method)
    with urllib.request.urlopen(request, timeout=10) as response: return response.read().decode()

assert json.loads(call('/health'))['status'] == 'ok'
assert json.loads(call('/v1/models', auth=True))['data'][0]['id'] == 'qwen-staged-mock'
payload = {'model': 'qwen-staged-mock', 'messages': [{'role': 'user', 'content': 'ping'}]}
assert 'healthy' in call('/v1/chat/completions', 'POST', payload, True)
assert 'data: [DONE]' in call('/v1/chat/completions', 'POST', {**payload, 'stream': True}, True)

# Set up cloudflared with guaranteed executable permission
cloudflared = Path('/tmp/cloudflared')
if not cloudflared.exists() or cloudflared.stat().st_size < 1000000:
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
cloudflared.chmod(0o755)

# Also copy to /kaggle/working/cloudflared and chmod if possible
try:
    shutil_target = work / 'cloudflared'
    if not shutil_target.exists():
        import shutil
        shutil.copy2(cloudflared, shutil_target)
    shutil_target.chmod(0o755)
except Exception:
    pass

tunnel_log = work / 'cloudflared.log'
tunnel_pid_path = work / 'cloudflared.pid'
tunnel_log_handle = tunnel_log.open('w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log_handle, stderr=subprocess.STDOUT, cwd=work)
tunnel_pid = tunnel.pid
tunnel_pid_path.write_text(str(tunnel_pid))

public_url = ''
for _ in range(30):
    time.sleep(1)
    log_text = tunnel_log.read_text(errors='replace') if tunnel_log.exists() else ''
    urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if urls:
        public_url = urls[-1]
        break

assert public_url, 'No tunnel URL found in log: ' + log_text[-2000:]
config = {'pid': pid, 'tunnel_pid': tunnel_pid, 'model': 'qwen-staged-mock', 'local_url': 'http://127.0.0.1:8000', 'base_url': public_url, 'api_key': key}
config_path.write_text(json.dumps(config, indent=2) + '\n')
config_path.chmod(0o600)

state_dir = work / '.qwen-state'
state_dir.mkdir(parents=True, exist_ok=True)
(state_dir / 'api_runtime.json').write_text(json.dumps({
    "stage": "api_runtime",
    "status": "ready",
    "model": "qwen-staged-mock",
    "base_url": public_url,
    "key": key
}, indent=2) + '\n')

print('TUNNEL_READY', config['base_url'])
print('API_READY', config['base_url'], config['model'])
print('API_STAGE_PASS health auth models chat stream')


TUNNEL_READY https://softball-engines-collectors-signal.trycloudflare.com
API_READY https://softball-engines-collectors-signal.trycloudflare.com qwen-staged-mock
API_STAGE_PASS health auth models chat stream


In [ ]:
import os, subprocess, urllib.request, shutil
from pathlib import Path

work = Path('/kaggle/working')
src = work / 'llama.cpp'
build = src / 'build'

def run(command, timeout=1800):
    print('RUN', ' '.join(map(str, command)))
    result = subprocess.run(command, cwd=work, text=True, capture_output=True, timeout=timeout)
    if result.stdout:
        print(result.stdout[-1000:])
    if result.stderr:
        print(result.stderr[-1000:])
    assert result.returncode == 0, f"Command failed with code {result.returncode}"
    return result

if not (src / '.git').exists():
    print("Cloning llama.cpp repository...")
    run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(src)])

if not (build / 'bin' / 'llama-cli').exists() or not (build / 'bin' / 'llama-server').exists():
    print("Preparing CUDA stubs...")
    stubs = work / 'cuda-stubs'
    stubs.mkdir(exist_ok=True)
    candidates = [
        '/usr/local/nvidia/lib64/libcuda.so.580.159.04',
        '/usr/local/cuda-12.8/compat/libcuda.so.570.124.06',
        '/usr/lib/x86_64-linux-gnu/libcuda.so.1',
        '/usr/local/cuda/lib64/libcuda.so.1'
    ]
    found = subprocess.run(['find', '/usr', '/opt', '/lib', '-name', 'libcuda.so*'], capture_output=True, text=True).stdout
    candidates += [line.strip() for line in found.splitlines() if line.strip()]
    driver = next((Path(item) for item in candidates if Path(item).exists()), None)
    assert driver, f"No libcuda found in {candidates}"
    print(f"Using CUDA driver library: {driver}")
    link = stubs / 'libcuda.so'
    link.unlink(missing_ok=True)
    link.symlink_to(driver)

    print("Configuring CMake for CUDA (architectures=75 for Tesla T4)...")
    run([
        'cmake', '-S', str(src), '-B', str(build),
        '-DGGML_CUDA=ON',
        '-DCMAKE_BUILD_TYPE=Release',
        '-DCMAKE_LIBRARY_PATH=' + str(stubs),
        '-DCMAKE_CUDA_ARCHITECTURES=75'
    ])
    print("Building llama-cli and llama-server with -j4...")
    run(['cmake', '--build', str(build), '--config', 'Release', '--target', 'llama-cli', 'llama-server', '-j4'])

cli = build / 'bin' / 'llama-cli'
server = build / 'bin' / 'llama-server'
assert cli.exists() and server.exists(), f"Binaries missing: cli={cli.exists()}, server={server.exists()}"

(work / 'llama-cli').unlink(missing_ok=True)
(work / 'llama-cli').symlink_to(cli)
(work / 'llama-server').unlink(missing_ok=True)
(work / 'llama-server').symlink_to(server)

subprocess.run(["chmod", "+x", str(cli), str(server)])
print("Testing llama-cli device discovery...")
devices = run([str(cli), '--list-devices'], timeout=120)
out = devices.stdout + devices.stderr
assert out.count('Tesla T4') >= 2, f"Expected 2 Tesla T4 devices, got: {out}"
print("Device discovery passed: 2 Tesla T4 devices confirmed.")

print("Downloading tiny-qwen validation model...")
model = work / 'tiny-qwen2.5-0.5b-instruct-q4_k_m.gguf'
url = 'https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_k_m.gguf?download=true'
if not model.exists() or model.stat().st_size < 10_000_000:
    urllib.request.urlretrieve(url, model)
assert model.stat().st_size > 10_000_000, model.stat().st_size
print(f"Model downloaded: {model.stat().st_size} bytes")

print("Running CUDA inference validation across both T4 GPUs...")
result = run([
    str(cli), '-m', str(model),
    '-p', 'Reply with exactly: LLAMA_CUDA_PASS',
    '-n', '16', '-ngl', '999',
    '--split-mode', 'layer', '--tensor-split', '1,1', '-st'
], timeout=300)
assert 'LLAMA_CUDA_PASS' in result.stdout, f"Unexpected response: {result.stdout}"
print("LLAMA_CPP_STAGE_PASS two T4 devices and prompt response")

from persistence import mark
mark('llama_cpp', devices='two T4', model=str(model), binary=str(server))
print("STAGE 2 COMPLETE & PERSISTED!")


In [ ]:
import json, os, subprocess, sys, time, urllib.request, re
from pathlib import Path

work = Path('/kaggle/working')
server_bin = work / 'llama-server'
model = work / 'tiny-qwen2.5-0.5b-instruct-q4_k_m.gguf'
key = os.environ.get('QWEN_API_KEY') or 'test-api-key-change-after-test'

subprocess.run(["chmod", "+x", str(server_bin)])
assert server_bin.exists(), "llama-server binary not found"
assert model.exists(), "model file not found"

# 1. Stop the mock HTTP server on port 8000
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
time.sleep(1)

# 2. Launch llama-server on port 8000
server_log = work / 'llama-server.log'
server_log_handle = server_log.open('w')

server_cmd = [
    str(server_bin),
    '-m', str(model),
    '--host', '127.0.0.1',
    '--port', '8000',
    '--api-key', key,
    '--alias', 'qwen2.5-0.5b',
    '-ngl', '999',
    '--split-mode', 'layer',
    '--tensor-split', '1,1',
    '-c', '4096',
    '-b', '256'
]
print("Starting llama-server:", ' '.join(server_cmd))
proc = subprocess.Popen(
    server_cmd,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
    cwd=work,
    preexec_fn=os.setsid
)

# 3. Wait for llama-server to be ready on /health
ready = False
for i in range(60):
    time.sleep(1)
    try:
        req = urllib.request.Request('http://127.0.0.1:8000/health')
        with urllib.request.urlopen(req, timeout=3) as resp:
            body = json.loads(resp.read().decode())
            if body.get('status') == 'ok':
                ready = True
                break
    except Exception:
        pass

assert ready, "llama-server failed to become ready: " + server_log.read_text(errors='replace')[-1500:]
print("llama-server is READY on http://127.0.0.1:8000!")

# 4. Verify locally
req = urllib.request.Request('http://127.0.0.1:8000/v1/models', headers={'Authorization': f'Bearer {key}'})
with urllib.request.urlopen(req, timeout=5) as resp:
    models_data = json.loads(resp.read().decode())
    print("Available models:", models_data)

# 5. Check cloudflared tunnel
tunnel_log = work / 'cloudflared.log'
log_text = tunnel_log.read_text(errors='replace') if tunnel_log.exists() else ''
urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
public_url = os.environ.get('TUNNEL_URL') or (urls[-1] if urls else '')

config_path = work / 'qwen_api.json'
config = {
    'pid': proc.pid,
    'model': 'qwen2.5-0.5b',
    'local_url': 'http://127.0.0.1:8000',
    'base_url': public_url,
    'api_key': key
}
config_path.write_text(json.dumps(config, indent=2) + '\n')

from persistence import mark
mark('small_qwen', status='ready', model='qwen2.5-0.5b', base_url=public_url)
print('SMALL_QWEN_STAGE_PASS', public_url)


In [ ]:
import json, os, re, subprocess, sys, time, urllib.request
from pathlib import Path

work = Path('/kaggle/working')
server = work / 'llama-server'
model = work / 'Qwen3.8-27B-ABLITERATED-Q4_K_M.gguf'
key = os.environ.get('QWEN_API_KEY') or 'test-api-key-change-after-test'
model_url = 'https://huggingface.co/Blackfrost-AI/Qwen3.8-27B-ABLITERATED-GGUF/resolve/main/Qwen3.8-27B-ABLITERATED-Q4_K_M.gguf?download=true'

subprocess.run(["chmod", "+x", str(server)])
assert server.exists(), "llama-server not found."

from persistence import mark, show
print('Checkpoints before final stage:', show())

if not model.exists() or model.stat().st_size < 10_000_000_000:
    print("Downloading Qwen3.8-27B model (16.8 GB)...")
    dl = subprocess.run(['curl', '-L', '-C', '-', model_url, '-o', str(model)], capture_output=True, text=True)
    if dl.returncode != 0:
        print("curl failed, trying aria2c...", dl.stderr[-500:])
        subprocess.run(['aria2c', '-x', '8', '-s', '8', '-k', '1M', '-c', model_url, '-o', model.name, '-d', str(work)], capture_output=True)

assert model.exists() and model.stat().st_size > 10_000_000_000, f"Model file incomplete: {model.stat().st_size if model.exists() else 0}"
print("Model ready:", model.stat().st_size, "bytes")

settings = [
    {'ctx': 8192, 'batch': 256, 'gpu_layers': 999},
    {'ctx': 4096, 'batch': 128, 'gpu_layers': 80},
    {'ctx': 2048, 'batch': 64, 'gpu_layers': 40}
]

# Stop previous server on port 8000
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
time.sleep(1)

working_setting = None
working_proc = None
server_log = work / 'llama-server-final.log'

for s in settings:
    print(f"Attempting configuration: ctx={s['ctx']}, batch={s['batch']}, gpu_layers={s['gpu_layers']}...")
    server_log_handle = server_log.open('w')
    cmd = [
        str(server),
        '-m', str(model),
        '--host', '127.0.0.1',
        '--port', '8000',
        '--api-key', key,
        '--alias', 'qwen-27b',
        '-ngl', str(s['gpu_layers']),
        '--split-mode', 'layer',
        '--tensor-split', '1,1',
        '-c', str(s['ctx']),
        '-b', str(s['batch'])
    ]
    proc = subprocess.Popen(cmd, stdout=server_log_handle, stderr=subprocess.STDOUT, cwd=work, preexec_fn=os.setsid)
    
    ready = False
    for _ in range(60):
        time.sleep(2)
        if proc.poll() is not None:
            break
        try:
            req = urllib.request.Request('http://127.0.0.1:8000/health')
            with urllib.request.urlopen(req, timeout=3) as r:
                if json.loads(r.read().decode()).get('status') == 'ok':
                    ready = True
                    break
        except Exception:
            pass
            
    if ready:
        print(f"SUCCESS! Running with ctx={s['ctx']}, batch={s['batch']}, gpu_layers={s['gpu_layers']}")
        working_setting = s
        working_proc = proc
        (work / 'final_qwen_command.sh').write_text('#!/bin/bash\n' + ' '.join(cmd) + '\n')
        break
    else:
        print("Configuration failed or OOM. Trying next setting...")
        try: os.killpg(os.getpgid(proc.pid), 9)
        except OSError: pass
        time.sleep(2)

assert working_setting, "Failed to run model under all candidate settings"

tunnel_log = work / 'cloudflared.log'
log_text = tunnel_log.read_text(errors='replace') if tunnel_log.exists() else ''
urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
public_url = os.environ.get('TUNNEL_URL') or (urls[-1] if urls else '')

config = {'pid': working_proc.pid, 'model': 'qwen-27b', 'local_url': 'http://127.0.0.1:8000', 'base_url': public_url, 'api_key': key, 'settings': working_setting}
(work / 'qwen_api.json').write_text(json.dumps(config, indent=2) + '\n')
mark('final_qwen', status='ready', model='qwen-27b', settings=working_setting, base_url=public_url)
print('FINAL_QWEN_STAGE_PASS', public_url, working_setting)


Checkpoints before final stage: ['api_runtime', 'final_qwen', 'gpu_stage', 'llama_cpp', 'persistence_test', 'small_qwen']


In [ ]:
import json, os, subprocess, time, urllib.request
from pathlib import Path
from persistence import completed, show, mark

work = Path('/kaggle/working')
state_dir = work / '.qwen-state'

print("=== 1. CHECKPOINT FILES IN .qwen-state ===")
assert state_dir.exists(), "State dir does not exist!"
for p in sorted(state_dir.glob("*.json")):
    print(f"FILE: {p.name}")
    print(p.read_text())

stages = ['api_runtime', 'gpu_stage', 'llama_cpp', 'small_qwen', 'final_qwen']
print("=== 2. STAGE COMPLETION VERIFICATION ===")
for st in stages:
    is_done = completed(st)
    print(f"Stage '{st}': {'[COMPLETED]' if is_done else '[MISSING]'}")
    assert is_done, f"Stage {st} is not completed!"

print("=== 3. ARTIFACTS IN /kaggle/working ===")
cli = work / 'llama-cli'
server = work / 'llama-server'
small_model = work / 'tiny-qwen2.5-0.5b-instruct-q4_k_m.gguf'
final_model = work / 'Qwen3.8-27B-ABLITERATED-Q4_K_M.gguf'
cmd_script = work / 'final_qwen_command.sh'
api_config = work / 'qwen_api.json'

assert cli.exists() and os.access(cli, os.X_OK), "llama-cli binary missing or not executable"
print(f"✓ llama-cli: {cli} (executable)")

assert server.exists() and os.access(server, os.X_OK), "llama-server binary missing or not executable"
print(f"✓ llama-server: {server} (executable)")

assert small_model.exists() and small_model.stat().st_size > 10_000_000, "small_model missing"
print(f"✓ small_model: {small_model.stat().st_size / 1e6:.1f} MB")

assert final_model.exists() and final_model.stat().st_size > 10_000_000_000, "final_model missing"
print(f"✓ final_model: {final_model.stat().st_size / 1e9:.2f} GB")

assert cmd_script.exists(), "final_qwen_command.sh missing"
print(f"✓ final_qwen_command.sh:\n{cmd_script.read_text().strip()}")

assert api_config.exists(), "qwen_api.json missing"
print(f"✓ qwen_api.json:\n{api_config.read_text().strip()}")

print("=== 4. TESTING RESTART RECOVERY ===")
cfg = json.loads(api_config.read_text())
key = cfg['api_key']

# Stop running server to simulate crash / restart
print("Stopping server to simulate restart...")
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
time.sleep(2)

# Verify stopped
stopped = False
try:
    urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=1)
except Exception:
    stopped = True
assert stopped, "Server did not stop"
print("✓ Server stopped confirmed.")

# Restart using persisted command script
print("Restarting server using persisted command script...")
restart_log = work / 'llama-server-restart.log'
restart_proc = subprocess.Popen(
    ['bash', str(cmd_script)],
    stdout=restart_log.open('w'),
    stderr=subprocess.STDOUT,
    cwd=work,
    preexec_fn=os.setsid
)

# Wait for recovery
restarted = False
for _ in range(30):
    time.sleep(2)
    try:
        r = urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2)
        if json.loads(r.read().decode()).get('status') == 'ok':
            restarted = True
            break
    except Exception:
        pass

assert restarted, "Server failed to recover on restart! Log:\n" + restart_log.read_text(errors='replace')[-1000:]
print("✓ Server recovered and healthy on http://127.0.0.1:8000!")

# Test completion on recovered server
req_data = json.dumps({
    "model": "qwen-27b",
    "messages": [{"role": "user", "content": "Reply with exactly: PERSISTENCE_RECOVERY_SUCCESS"}],
    "max_tokens": 64
}).encode()
req = urllib.request.Request(
    'http://127.0.0.1:8000/v1/chat/completions',
    data=req_data,
    headers={'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'}
)
with urllib.request.urlopen(req, timeout=30) as resp:
    result = json.loads(resp.read().decode())
    print("✓ Model response on recovered server:", result['choices'][0]['message'])

# Update PID in config
cfg['pid'] = restart_proc.pid
api_config.write_text(json.dumps(cfg, indent=2) + '\n')
mark('persistence_test', status='verified', timestamp=time.time())

print("\nPERSISTENCE_TEST_PASS: All checkpoints, artifacts, and restart recovery verified!")
